---
title: '性能模型 · Roofline 与 CPU 向量加法'
---


# 性能模型：从解释、预测到计时

**状态：教学 Notebook 重写，新增代码未运行。** [历史 CPU 输出](recorded-cpu.ipynb)保留真实记录与旧代码来源，不代表你的服务器。
本页把概念、公式、图和实验代码放在一起，按 **讲解 → 预测 → 代码 → 观察** 阅读。

要解释：计算量很少的操作为什么仍可能很慢？减少算术工作为什么不一定得到同等加速？

在远程服务器准备本主题环境并选择 kernel：

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r topics/foundations/performance-model/requirements.txt
```

工作目录为本 Notebook 所在目录；VS Code Remote SSH 已配置 `${fileDirname}`。


## 1 · 算术与数据移动同时约束执行

做 `out = x + y` 时，需要读入两个输入、执行加法并写出结果。
假设工作能够足够充分地利用硬件，算术吞吐和指定内存层级的数据移动分别给出时间下界；取两者的较大值形成一个简化上界模型。

```mermaid
---
config:
  theme: base
  themeVariables:
    background: '#FAF8F4'
    primaryColor: '#E4F0EB'
    primaryTextColor: '#292524'
    primaryBorderColor: '#0F766E'
    lineColor: '#6B635A'
    fontFamily: 'system-ui, sans-serif'
---
flowchart LR
    input[Input buffers]:::memory --> transfer[Data movement]:::communication
    transfer --> compute[Vector addition]:::compute
    compute --> output[Output buffer]:::memory
    classDef compute fill:#EAF0F8,stroke:#315C8C,color:#292524
    classDef memory fill:#F8EFDA,stroke:#9A6700,color:#292524
    classDef communication fill:#E4F0EB,stroke:#0F766E,color:#292524
    classDef warning fill:#F9E7E3,stroke:#B54735,color:#292524
```

图：向量加法的简化数据流，省略真实缓存层级、调度、同步与启动成本。

**预测：** 如果访存约束更紧，计算峰值翻倍会改变哪一项上界？


In [ ]:
# 这是教学模型的计算，不是机器实测。
flops = 1_000_000
useful_bytes = 12_000_000
peak_flops_per_second = 1_000_000_000_000
bandwidth_bytes_per_second = 100_000_000_000
compute_floor_s = flops / peak_flops_per_second
memory_floor_s = useful_bytes / bandwidth_bytes_per_second
print({"compute_lower_bound_s": compute_floor_s, "memory_lower_bound_s": memory_floor_s,
       "model_time_lower_bound_s": max(compute_floor_s, memory_floor_s)})


## 2 · Roofline：先指定流量属于哪个内存层级

算术强度为 $I=F/B$，其中 $B$ 是指定层级的数据流量。若计算峰值为 $P_{peak}$、带宽为 $BW$，

$$P_{roof}=\min(P_{peak}, BW\cdot I).$$

两条线在 ridge point $I=P_{peak}/BW$ 相交。它是上界，不承诺真实操作达到上界，也没有包含所有启动、依赖、同步与调度开销。
[Berkeley Lab 的 Roofline 说明](https://amcr.lbl.gov/departments/computer-science-department/ppan/roofline-performance-model/)

**预测：** 教学带宽 100 GB/s、峰值 1000 GFLOP/s，将强度从 1 提到 10，再到 100，上界怎样变化？


In [ ]:
BANDWIDTH_GB_S = 100.0
PEAK_GFLOP_S = 1000.0
INTENSITIES = [1.0, 10.0, 100.0]

for intensity in INTENSITIES:
    print({"intensity_FLOP_per_byte": intensity,
           "roof_GFLOP_per_second": min(PEAK_GFLOP_S, BANDWIDTH_GB_S * intensity)})
print("Ridge point (FLOP/byte):", PEAK_GFLOP_S / BANDWIDTH_GB_S)


### 改参数，用代码重画模型

把上面的带宽或峰值改成另一组教学值，先预测折点与上界，再重跑下方单元。共享样式和语义颜色从仓库配置读取。
模型图没有绑定任何 CPU/GPU 型号；蓝色为计算上界，赭黄为访存上界，绿色为两个约束共同给出的 Roofline。


In [ ]:
import csv
import json
import os
import platform
import subprocess
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

TOPIC = Path.cwd().resolve()
assert (TOPIC / "lab.ipynb").is_file(), "请把 kernel 工作目录切换到性能模型主题"
REPO = TOPIC.parents[2]
plt.style.use(REPO / "assets/styles/plots.mplstyle")
colors = json.loads((REPO / "assets/styles/tokens.json").read_text())["colors"]
RUN_DIR = REPO / ".runs" / "performance-notebook" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
intensity_axis = np.logspace(-2, 3, 200)

fig, ax = plt.subplots()
ax.loglog(intensity_axis, np.full_like(intensity_axis, PEAK_GFLOP_S),
          color=colors["compute"], linestyle="--", label="Compute ceiling")
ax.loglog(intensity_axis, BANDWIDTH_GB_S * intensity_axis,
          color=colors["memory"], linestyle=":", label="Bandwidth ceiling")
ax.loglog(intensity_axis, np.minimum(PEAK_GFLOP_S, BANDWIDTH_GB_S * intensity_axis),
          color=colors["communication"], label="Roofline upper bound")
ax.set(title="Roofline · teaching model", xlabel="Arithmetic intensity (FLOP/byte)",
       ylabel="Performance upper bound (GFLOP/s)")
ax.legend()
fig.savefig(RUN_DIR / "roofline-model.svg", bbox_inches="tight")
fig.savefig(RUN_DIR / "roofline-model.png", bbox_inches="tight")
plt.show()


**复述：** 什么时候强度提高十倍，上界也提高十倍？什么时候停止？一个实测点远低于上界，下一步该补哪一种观测？

## 3 · 从流量估计到真实 CPU 向量加法

固定 float32，预分配输出，使用 `np.add(x, y, out=out)`。读两个输入、写一个输出得到 $B_{useful}=12N$，
但这个 useful bytes 不等于硬件 DRAM 流量：缓存命中、写分配等都会改变真实数据路径。
有效带宽为 $B_{useful}/t$，用十进制 GB/s；下面每个规模预热 3 次、计时 25 次。
[NumPy add](https://numpy.org/doc/stable/reference/generated/numpy.add.html) · [perf_counter_ns](https://docs.python.org/3/library/time.html#time.perf_counter_ns)

**预测：** 小输入与大输入的有效带宽可能有什么变化？为什么不能把最高值当作 DRAM 峰值？


In [ ]:
SIZES = [1024, 16384, 262144, 4194304]
WARMUPS = 3
REPEATS = 25
record = {
    "kind": "measured CPU vector addition from Notebook", "status": "running",
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "environment": {"platform": platform.platform(), "python": platform.python_version(),
        "numpy": np.__version__, "host_cpu_count": os.cpu_count(),
        "git_revision": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True, capture_output=True, check=True).stdout.strip(),
        "git_status": subprocess.run(["git", "status", "--porcelain"], cwd=REPO, text=True, capture_output=True, check=True).stdout.splitlines()},
    "measurement": {"sizes": SIZES, "warmups": WARMUPS, "repeats": REPEATS,
                    "dtype": "float32", "clock": "time.perf_counter_ns", "useful_bytes_model": "12 * elements"},
    "results": [],
}

def save_record():
    (RUN_DIR / "run.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")

save_record()
print(json.dumps(record["environment"], indent=2))


### 把被计时的操作展开在单元格里

输入和输出分配在计时之外；计时区间只包含调用本身及计时开销。每次计时后保存原始样本，检查输出为 3。
正确性、环境记录、预热与重复测量都是解释性能的前提；不固定断言机器相关的排序。


In [ ]:
def measure_add(size):
    x = np.ones(size, dtype=np.float32)
    y = np.full(size, 2, dtype=np.float32)
    out = np.empty_like(x)
    for _ in range(WARMUPS):
        np.add(x, y, out=out)
    samples_ns = []
    for _ in range(REPEATS):
        started = time.perf_counter_ns()
        np.add(x, y, out=out)
        samples_ns.append(time.perf_counter_ns() - started)
    assert np.all(out == 3), "向量加法结果不正确"
    median_ns = float(np.median(samples_ns))
    useful_bytes = 3 * size * x.itemsize
    return {"elements": size, "useful_bytes": useful_bytes, "samples_ns": samples_ns,
            "median_ns": median_ns, "p25_ns": float(np.percentile(samples_ns, 25)),
            "p75_ns": float(np.percentile(samples_ns, 75)), "effective_gb_s": useful_bytes / median_ns}


现在运行每个规模。与历史记录比较时，要先核对机器、版本、输入与计时方法。
运行失败会保留部分样本与 traceback，不能把失败运行当作完整性能结论。


In [ ]:
try:
    for size in SIZES:
        record["results"].append(measure_add(size))
        save_record()
    with (RUN_DIR / "summary.csv").open("w", newline="", encoding="utf-8") as file:
        columns = [key for key in record["results"][0] if key != "samples_ns"]
        writer = csv.DictWriter(file, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(record["results"])
    record["status"] = "passed"
except BaseException as error:
    record["status"] = "failed"
    record["error"] = {"type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()}
    raise
finally:
    save_record()

print("Recorded:", RUN_DIR)
print([(row["elements"], row["effective_gb_s"]) for row in record["results"]])


## 4 · 画真实样本，不替结果编故事

点由中位计时换算，误差棒用第 25–75 百分位计时换算得到的有效带宽区间；它不是置信区间。
这张图使用本次 `record` 的原始样本，不读取历史机器的值来填充新实验。


In [ ]:
rows = record["results"]
sizes = np.array([row["elements"] for row in rows])
useful_bytes = np.array([row["useful_bytes"] for row in rows])
bandwidth = np.array([row["effective_gb_s"] for row in rows])
lower = useful_bytes / np.array([row["p75_ns"] for row in rows])
upper = useful_bytes / np.array([row["p25_ns"] for row in rows])

fig, ax = plt.subplots()
ax.errorbar(sizes, bandwidth, yerr=np.vstack([bandwidth - lower, upper - bandwidth]),
            color=colors["memory"], marker="o", capsize=4, label="Median; timing interquartile range")
ax.set_xscale("log")
ax.set(title="CPU vector addition · measured", xlabel="Array length (elements)",
       ylabel="Effective bandwidth (GB/s)")
ax.legend()
fig.savefig(RUN_DIR / "vector-add-effective-bandwidth.svg", bbox_inches="tight")
fig.savefig(RUN_DIR / "vector-add-effective-bandwidth.png", bbox_inches="tight")
plt.show()


## 5 · 从证据修正解释

此结果描述记录环境中的 NumPy CPU 操作。缓存、调用与计时开销、共享负载和频率变化都会影响观察，不能把它套成 GPU 或 DRAM 峰值。

- **运行前预测：** 待填写。
- **实际观察：** 新代码未运行，待填写。
- **原先解释 → 新证据 → 修正后的解释：** 待填写。
- **模型遗漏的前提及下一步观测：** 待填写。

保存 Notebook 的真实输出、运行 JSON、CSV 与图。在服务器回收前取回新临时目录，比较源码和说明差异后，再精选到 `results/` 与 `assets/`。
剩余问题记到 [GAPS](../../../GAPS.md)。[历史记录](recorded-cpu.ipynb)保留原先机器上的样本与来源；不要用它代替本次实验。

来源：[Berkeley Lab Roofline](https://amcr.lbl.gov/departments/computer-science-department/ppan/roofline-performance-model/)、[NumPy add](https://numpy.org/doc/stable/reference/generated/numpy.add.html)、[Python timing](https://docs.python.org/3/library/time.html#time.perf_counter_ns)。资料查阅日期：2026-10-02。
